# Ejercicio 4 — Análisis exploratorio con DuckDB

Datos: viajes de taxis amarillos y verdes de enero a agosto de 2026, leídos directamente de los archivos
Parquet mediante la vista `trips_clean` (reglas de calidad del Ejercicio 3). Las consultas están en
`sql/04_eda/` y la documentación completa en `docs/04_eda.md`.

DuckDB hace todas las agregaciones sobre los 28.9 millones de viajes; a pandas y Matplotlib solo llegan
los resultados ya resumidos (decenas o cientos de filas).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "scripts"))

import matplotlib.pyplot as plt
import pandas as pd

import charts
import taxi_db

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)
charts.apply_style()
con = taxi_db.connect()

## 4.1 Preguntas

| # | Pregunta | Dimensión |
|---|---|---|
| P1 | ¿Cómo cambia el volumen diario de viajes de un mes a otro y cuánto ingreso generan? | Temporal |
| P2 | ¿En qué horas y días se concentra la demanda de cada tipo de taxi? | Temporal |
| P3 | ¿Cómo es un viaje típico (distancia, duración, velocidad, monto, pasajeros)? | Características |
| P4 | ¿Cómo se distribuyen las distancias? | Distribuciones |
| P5 | ¿Cómo se distribuye el monto total pagado? | Distribuciones |
| P6 | ¿En qué distritos inician los viajes de cada tipo? | Amarillo vs. verde |
| P7 | ¿Cuáles son las zonas con más abordajes? | Amarillo vs. verde |
| P8 | ¿Cómo pagan los pasajeros y cuánto pagan con cada medio? | Pago |
| P9 | ¿Cuánta propina se deja con tarjeta y varía según la hora? | Pago |
| P10 | ¿Qué peso tienen los viajes al aeropuerto en volumen e ingresos? | Características |
| P11 | ¿A qué proporción de viajes se le cobra el cargo de congestión de Manhattan (CBD)? | Pago |
| P12 | ¿Cuántos valores atípicos quedan después de la limpieza? | Atípicos |
| P13 | ¿Los proveedores reportan los montos de forma consistente? | Inconsistencias |

## P1 — Volumen mensual

Se normaliza por días con viajes (`trips_per_day`) porque los meses tienen distinta duración.

In [ ]:
monthly = taxi_db.show(con, "04_eda/01_monthly_volume.sql")
monthly

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, taxi in zip(axes, ["yellow", "green"]):
    data = monthly[monthly.taxi_type == taxi]
    ax.plot(data.month, data.trips_per_day, marker="o", color=charts.TAXI_COLORS[taxi])
    ax.set_title(f"Taxi {charts.TAXI_LABELS[taxi].lower()}: viajes por día")
    ax.set_xticks(range(1, 9), charts.MONTHS[:8])
    ax.set_ylim(0, data.trips_per_day.max() * 1.15)
    charts.thousands(ax)
    peak = data.loc[data.trips_per_day.idxmax()]
    ax.annotate(f"{peak.trips_per_day:,.0f}", (peak.month, peak.trips_per_day), textcoords="offset points", xytext=(0, 8), ha="center")
plt.tight_layout()
plt.show()

## P2 — Demanda por hora y día de la semana

Promedio de viajes por hora para cada día de la semana (1 = lunes).

In [ ]:
hourly = taxi_db.show(con, "04_eda/02_hour_weekday.sql")
hourly

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, taxi in zip(axes, ["yellow", "green"]):
    grid = hourly[hourly.taxi_type == taxi].pivot(index="hour", columns="weekday", values="trips_per_day")
    image = ax.imshow(grid, aspect="auto", cmap=charts.SEQUENTIAL_MAP)
    ax.set_title(f"Taxi {charts.TAXI_LABELS[taxi].lower()}: viajes promedio por hora")
    ax.set_xticks(range(7), charts.WEEKDAYS)
    ax.set_yticks(range(0, 24, 2))
    ax.set_ylabel("Hora de abordaje")
    ax.grid(False)
    fig.colorbar(image, ax=ax, label="viajes por hora")
plt.tight_layout()
plt.show()

## P3 — Perfil de un viaje típico

Se usan medianas y percentil 90 porque las distribuciones tienen colas largas.

In [ ]:
profile = taxi_db.show(con, "04_eda/03_trip_profile.sql")
profile

## P4 — Distribución de distancias

In [ ]:
distance = taxi_db.show(con, "04_eda/04_distance_distribution.sql")
distance

In [ ]:
buckets = distance.miles.unique()
positions = range(len(buckets))
width = 0.4
fig, ax = plt.subplots()
for offset, taxi in [(-width / 2, "yellow"), (width / 2, "green")]:
    data = distance[distance.taxi_type == taxi].set_index("miles").reindex(buckets)
    ax.bar([p + offset for p in positions], data.pct_of_type, width=width - 0.04, color=charts.TAXI_COLORS[taxi], label=charts.TAXI_LABELS[taxi])
ax.set_xticks(list(positions), buckets)
ax.set_xlabel("Distancia (millas)")
ax.set_ylabel("% de los viajes del tipo")
ax.set_title("Distribución de distancias por tipo de taxi")
ax.legend()
plt.show()

## P5 — Distribución del monto total

Intervalos de 5 dólares; el último agrupa los viajes de 150 o más.

In [ ]:
fares = taxi_db.show(con, "04_eda/05_fare_distribution.sql")
fares

In [ ]:
fig, ax = plt.subplots()
for taxi in ["yellow", "green"]:
    data = fares[fares.taxi_type == taxi]
    ax.plot(data.total_bucket, data.pct_of_type, drawstyle="steps-post", color=charts.TAXI_COLORS[taxi], label=charts.TAXI_LABELS[taxi])
ax.axvspan(80, 105, color="#e4e3df", alpha=0.6, linewidth=0)
ax.annotate("JFK: tarifa fija de 70 USD
más recargos y peajes", (92.5, 4), ha="center", color="#52514e")
ax.set_xlabel("Monto total (USD)")
ax.set_ylabel("% de los viajes del tipo")
ax.set_title("Distribución del monto total pagado")
ax.legend()
plt.show()

## P6 — Distrito de abordaje

In [ ]:
boroughs = taxi_db.show(con, "04_eda/06_borough_by_type.sql")
boroughs

In [ ]:
order = ["Manhattan", "Queens", "Brooklyn", "Bronx"]
positions = range(len(order))
fig, ax = plt.subplots()
for offset, taxi in [(-0.2, "yellow"), (0.2, "green")]:
    data = boroughs[boroughs.taxi_type == taxi].set_index("pickup_borough").reindex(order)
    bars = ax.barh([p + offset for p in positions], data.pct_of_type, height=0.36, color=charts.TAXI_COLORS[taxi], label=charts.TAXI_LABELS[taxi])
    ax.bar_label(bars, fmt="%.1f %%", padding=3, color="#52514e")
ax.set_yticks(list(positions), order)
ax.invert_yaxis()
ax.set_xlabel("% de los viajes del tipo")
ax.set_title("Distrito donde inicia el viaje")
ax.legend(loc="lower right")
plt.show()

## P7 — Zonas con más abordajes

In [ ]:
zones_top = taxi_db.show(con, "04_eda/07_top_pickup_zones.sql")
zones_top

## P8 — Medio de pago

In [ ]:
payments = taxi_db.show(con, "04_eda/08_payment_mix.sql")
payments

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, taxi in zip(axes, ["yellow", "green"]):
    data = payments[payments.taxi_type == taxi].sort_values("pct_of_type")
    bars = ax.barh(data.payment, data.pct_of_type, color=charts.TAXI_COLORS[taxi])
    ax.bar_label(bars, fmt="%.1f %%", padding=3, color="#52514e")
    ax.set_title(f"Taxi {charts.TAXI_LABELS[taxi].lower()}: medio de pago")
    ax.set_xlabel("% de los viajes")
    ax.set_xlim(0, 80)
plt.tight_layout()
plt.show()

## P9 — Propinas con tarjeta por hora

Solo pagos con tarjeta (`payment_type = 1`): las propinas en efectivo no se registran.

In [ ]:
tips = taxi_db.show(con, "04_eda/09_tip_behavior.sql")
tips

In [ ]:
fig, ax = plt.subplots()
for taxi in ["yellow", "green"]:
    data = tips[tips.taxi_type == taxi]
    ax.plot(data.hour, data.tip_pct_of_fare, marker="o", color=charts.TAXI_COLORS[taxi], label=charts.TAXI_LABELS[taxi])
ax.set_xticks(range(0, 24, 2))
ax.set_xlabel("Hora de abordaje")
ax.set_ylabel("Propina como % de la tarifa")
ax.set_title("Propina con tarjeta según la hora")
ax.legend()
plt.show()

## P10 — Viajes al aeropuerto

Un viaje es «de aeropuerto» si inicia o termina en una zona de servicio `Airports` (JFK, LaGuardia) o `EWR` (Newark).

In [ ]:
airports = taxi_db.show(con, "04_eda/10_airport_trips.sql")
airports

In [ ]:
data = airports[airports.trip_kind == "Aeropuerto"]
fig, ax = plt.subplots(figsize=(8, 4))
positions = range(len(data))
bars_trips = ax.bar([p - 0.2 for p in positions], data.pct_trips, width=0.36, color="#9fc4ee", label="% de viajes")
bars_revenue = ax.bar([p + 0.2 for p in positions], data.pct_revenue, width=0.36, color="#2a78d6", label="% de ingresos")
ax.bar_label(bars_trips, fmt="%.1f %%", padding=3, color="#52514e")
ax.bar_label(bars_revenue, fmt="%.1f %%", padding=3, color="#52514e")
ax.set_xticks(list(positions), [charts.TAXI_LABELS[t] for t in data.taxi_type])
ax.set_title("Peso de los viajes de aeropuerto")
ax.legend()
plt.show()

## P11 — Cargo de congestión de Manhattan (CBD)

In [ ]:
cbd = taxi_db.show(con, "04_eda/11_cbd_congestion_fee.sql")
cbd

In [ ]:
fig, ax = plt.subplots()
for taxi in ["yellow", "green"]:
    data = cbd[cbd.taxi_type == taxi]
    ax.plot(data.month, data.pct_with_cbd_fee, marker="o", color=charts.TAXI_COLORS[taxi], label=charts.TAXI_LABELS[taxi])
ax.set_xticks(range(1, 9), charts.MONTHS[:8])
ax.set_ylim(0, 100)
ax.set_ylabel("% de viajes con cargo CBD")
ax.set_title("Viajes a los que se cobró el cargo de congestión")
ax.legend()
plt.show()

## P12 — Valores atípicos

Regla IQR sobre el monto total (Q3 + 1.5·IQR), velocidades imposibles y tarifas por milla extremas.

In [ ]:
outliers = taxi_db.show(con, "04_eda/12_outliers.sql")
outliers

## P13 — Consistencia de montos por proveedor

Porcentaje de viajes en que la suma de componentes coincide con `total_amount`, incluyendo o no los recargos de congestión, aeropuerto y CBD.

In [ ]:
vendors = taxi_db.show(con, "04_eda/13_vendor_consistency.sql")
vendors

## 4.5 Hallazgos

1. **Los viajes de aeropuerto valen el triple.** Son el 8.2 % de los viajes amarillos, pero generan el
   21.0 % de los ingresos (promedio de 77.95 USD contra 25.96 USD de un viaje urbano). En la distribución
   de montos aparece una meseta entre 80 y 105 USD que corresponde a la tarifa fija de 70 USD entre JFK y
   Manhattan más recargos, peajes y propina.
2. **Amarillos y verdes atienden mercados distintos.** El 86.7 % de los viajes amarillos empieza en
   Manhattan, sobre todo en Midtown y el Upper East Side, y al 71–77 % se le cobra el cargo de congestión
   CBD. Los verdes empiezan en el norte de Manhattan (East Harlem concentra el 40 %), Queens y Brooklyn, y
   solo al 7–10 % se le cobra el cargo CBD.
3. **El taxi amarillo es nocturno y de fin de semana; el verde es de horario laboral.** El amarillo tiene
   su pico entre las 17 y las 22 horas y los viernes y sábados mantiene más de 5,000 viajes por hora hasta
   la 1 a. m.; el verde sube a las 7–9 y a las 16–18 de lunes a viernes, con un patrón de traslado al
   trabajo.
4. **La demanda cae en verano.** Los viajes amarillos por día subieron de 114,854 en enero a 127,844 en
   mayo y bajaron a 103,349 en agosto (−19 % respecto de mayo), mientras el monto promedio se mantuvo
   estable cerca de 30 USD.
5. **La calidad depende del proveedor.** El 24.6 % de los viajes amarillos son registros *flex fare*
   (`payment_type = 0`) sin pasajeros ni tarifa; el proveedor 7 no registra la hora de descenso, y la suma
   de componentes coincide con el total en el 76 % de los viajes del proveedor 2, pero solo en el 14 % de
   los del proveedor 1, que incluye los recargos dentro de `extra`.